# PDB Graph Store usage exemple

## Installing dependences

In [1]:
!sudo apt update > /dev/null; sudo apt install -y gcc > /dev/null

In [2]:
!pip install -r ./../r.txt > /dev/null

## Importing graphein modules

In [1]:
from graphein.protein.config import ProteinGraphConfig
from graphein.protein.graphs import construct_graph
from graphein.protein.utils import download_pdb

## Importing graph store modules

In [2]:
from pkg.PDBGraphStore import PDBGraphStore

In [5]:
from pkg.operations import remove_graph_from_store
from pkg.operations import split_graph_store
from pkg.operations import merge_graph_stores

ModuleNotFoundError: No module named 'PDBGraphStore'

In [3]:
import pkg.edge_functions_Model as edgeModel

## Importing system modules

In [4]:
import traceback
import os

## Reading dataset toy.txt at ./../data/

In [5]:
def read_dataset(general_data_path, dataset_txt_name, file_mode='r'):
    pdb_codes = list()
    with open(f"{general_data_path}/{dataset_txt_name}", file_mode) as file:
        for line in file:
            if line[0] != '#':
                pdb_codes.append(line.strip().upper())

    return pdb_codes

pdb_codes = read_dataset("./../data", "ligand_PLP.txt", 'r')
pdb_codes

['1B54',
 '1B8G',
 '1BJ4',
 '1C7G',
 '1F2D',
 '1FC4',
 '1HKV',
 '1I2K',
 '1JNW',
 '1JS3',
 '1KL7',
 '1LK9',
 '1LKC',
 '1M54',
 '1N31',
 '1O61',
 '1ORD',
 '1RFU',
 '1T3I',
 '1TDJ',
 '1UIM',
 '1V72',
 '1WYU',
 '1XRS',
 '1YGP',
 '2ABJ',
 '2AQ6',
 '2CFT',
 '2CTZ',
 '2DGM',
 '2E7I',
 '2FYF',
 '2HZP',
 '2JIS',
 '2UZP',
 '2Z67',
 '2ZY4',
 '3ANV',
 '3BB8',
 '3E77',
 '3F9T',
 '3HQT',
 '3I16',
 '3IF2',
 '3IHJ',
 '3KOW',
 '3LY1',
 '3MEB',
 '3N29',
 '3N2O',
 '3O05',
 '3PC2',
 '3SS7',
 '4BF5',
 '4CMD',
 '4H6D',
 '4M2J',
 '5DDS',
 '5HSJ',
 '5HXX',
 '5IJG',
 '5W71',
 '5X03']

## Defining the edge functions

In [6]:
def define_graphein_edge_funcs():
   return [edgeModel.edge_functions_dict[f] for f in ["delaunay", "aromatic", "aromatic_sulphur"]]

edge_funcs = define_graphein_edge_funcs()

edge_funcs

[<function graphein.protein.edges.distance.add_delaunay_triangulation(G: 'nx.Graph', allowable_nodes: 'Optional[List[str]]' = None)>,
 <function graphein.protein.edges.distance.add_aromatic_interactions(G: 'nx.Graph', pdb_df: 'Optional[pd.DataFrame]' = None)>,
 <function graphein.protein.edges.distance.add_aromatic_sulphur_interactions(G: 'nx.Graph', rgroup_df: 'Optional[pd.DataFrame]' = None)>]

## Defining graph configuration (with edge functions and node granularity)

In [7]:
def define_configuration(edge_construction_funcs):
    return {
        "granularity": "CA",
        "edge_construction_functions": edge_construction_funcs
    }

graph_config = ProteinGraphConfig(**define_configuration(edge_funcs))

graph_config

ProteinGraphConfig(granularity='CA', keep_hets=[], insertions=True, alt_locs='max_occupancy', pdb_dir=None, verbose=False, exclude_waters=True, deprotonate=False, protein_df_processing_functions=None, edge_construction_functions=[<function add_delaunay_triangulation at 0x7ff7a3c22d40>, <function add_aromatic_interactions at 0x7ff7a3c22700>, <function add_aromatic_sulphur_interactions at 0x7ff7a3c227a0>], node_metadata_functions=[<function meiler_embedding at 0x7ff7a3c29300>], edge_metadata_functions=None, graph_metadata_functions=None, get_contacts_config=None, dssp_config=None)

In [8]:
def get_pdb_file(pdb_data_path, pdb_code):
    pdb_code = pdb_code.lower()
    if os.path.exists(f"{pdb_data_path}/{pdb_code}.pdb"):
        print(f"Reading {pdb_code} from local directory")
        pdb_file = os.path.abspath(f"{pdb_data_path}/{pdb_code}.pdb")
    else:
        print(f"Downloading {pdb_code} from PDB")
        try:
            pdb_file = download_pdb(pdb_code, f"{pdb_data_path}/")
        except Exception as e:
            raise e

    if pdb_file == None:
        raise Exception("Error reading the pdb file")

    return pdb_file

In [9]:
def prepare_graph(pdb_data_path, pdb_code):
    protein_graphs = dict()

    try:
        pdb_file = get_pdb_file(pdb_data_path=pdb_data_path, pdb_code=pdb_code)
    except Exception as e:
        raise e

    graph = construct_graph(config=graph_config, path=pdb_file)
    graph.graph["pdb_code"] = pdb_code
    print(graph, "\n")

    config = graph.graph["config"]
    graph_pdb_code = graph.graph["pdb_code"]
    graph.graph.clear()
    graph.graph["config"] = config
    graph.graph["pdb_code"] = graph_pdb_code


    protein_graphs[pdb_code] = graph.copy()

    return protein_graphs

In [15]:
def build_pdb_store():
    n_nodes = 0
    n_edges = 0
    pdb_store = PDBGraphStore()

    for _, pdb_code in enumerate(pdb_codes.copy()):
        try:
            graph = prepare_graph("./../data/pdb_files", pdb_code)
            n_nodes += len(graph[pdb_code].nodes)
            n_edges += len(graph[pdb_code].edges)
        except Exception as e:
            msg = traceback.format_exc()
            print(msg)
            continue
        pdb_store.insert(graph)
    
    del graph

    return pdb_store, n_nodes, n_edges

In [16]:
pdb_graph_store, n_nodes, n_edges = build_pdb_store()
pdb_graph_store

Output()

Reading 1b54 from local directory


Output()

Graph named '1b54' with 230 nodes and 1612 edges 

Reading 1b8g from local directory


Output()

...

Output()

Graph named '5ijg' with 756 nodes and 5604 edges 

Reading 5w71 from local directory


Output()

Graph named '5w71' with 822 nodes and 6159 edges 

Reading 5x03 from local directory


Graph named '5x03' with 728 nodes and 5400 edges 



## Playing with pdb graph store object

In [17]:
print(len(pdb_graph_store.get_body_parts()['node_label_to_node_id']))
print(n_nodes)

35651
67803


In [18]:
print(len(pdb_graph_store.get_body_parts()['edge_label_to_edge_id']))
print(n_edges)

500700
510770


#### Getting the list of pdb codes in this object

In [14]:
print(pdb_graph_store.get_pdb_list())

dict_keys(['1BXL', '1G5J', '1TY4', '1ZY3', '2A5Y', '2BZW', '2JM6', '2K7W', '2KBW', '2LP8', '2LR1', '2M04', '2M5B', '2MEJ', '2NL9'])


#### Extracting one pdb

In [15]:
g = pdb_graph_store.extract('1bxl')

print(g.graph['pdb_code'])

1BXL


#### Removing one pdb

In [16]:
pdb_graph_store = remove_graph_from_store(['1BXL'], pdb_graph_store)

print(pdb_graph_store)

PDBGraphStore with 14 pdbs


In [17]:
print(pdb_graph_store.get_pdb_list())

dict_keys(['1G5J', '1TY4', '1ZY3', '2A5Y', '2BZW', '2JM6', '2K7W', '2KBW', '2LP8', '2LR1', '2M04', '2M5B', '2MEJ', '2NL9'])


#### Inserting one pdb

In [18]:
pdb_graph_store.insert({"1BXL": g})

print(pdb_graph_store)

PDBGraphStore with 15 pdbs


In [19]:
print(pdb_graph_store.get_pdb_list())

dict_keys(['1G5J', '1TY4', '1ZY3', '2A5Y', '2BZW', '2JM6', '2K7W', '2KBW', '2LP8', '2LR1', '2M04', '2M5B', '2MEJ', '2NL9', '1BXL'])


#### Spliting pdb store into 2 distincts stores

In [20]:
store1, store2 = split_graph_store(pdb_graph_store, ["1BXL", "1G5J"])

In [21]:
print(store1.get_pdb_list())
print(store2.get_pdb_list())

dict_keys(['1G5J', '1BXL'])
dict_keys(['1TY4', '1ZY3', '2A5Y', '2BZW', '2JM6', '2K7W', '2KBW', '2LP8', '2LR1', '2M04', '2M5B', '2MEJ', '2NL9'])


In [24]:
merged = merge_graph_stores([store1, store2])

print(merged.get_pdb_list())
print(len(merged.get_pdb_list()))

{'config_CA_[]_True_max_occupancy_None_False_True_False_None_[<function add_delaunay_triangulation at 0x7fc668df3920>, <function add_aromatic_interactions at 0x7fc668df32e0>, <function add_aromatic_sulphur_interactions at 0x7fc668df3380>]_[<function meiler_embedding at 0x7fc668df5ee0>]_None_None_None_None'}
dict_keys(['1TY4', '1ZY3', '2A5Y', '2BZW', '2JM6', '2K7W', '2KBW', '2LP8', '2LR1', '2M04', '2M5B', '2MEJ', '2NL9', '1BXL', '1G5J'])
15
